In [ ]:
%run "./00_CC_Mapping_Setup.ipynb"

In [ ]:
%sql
/* ===================================================================================
   METRIC: EBA04 - Gifts/Travel or Lodging Expenses_Non POs
   
   BUSINESS QUESTION: 
   During the assessment period, how many instances were recorded by the Unit where 
   employees provide or receive gifts or paid for travel/entertainment to Non POs 
   exceeding the $250 threshold?
   
   LOGIC & ARCHITECTURE SUMMARY:
   1. MASTER AU ANCHOR: Uses the Master AU list as the absolute key for the output. 
   2. COUPA DATA ONLY: Unions the 7 Coupa files, pulling the 'Total' column.
   3. ACCOUNT PARSING: Splits the Account string to extract the Cost Center (index 0) 
      and the Category Code (index 2).
   4. THRESHOLD RULE: Strips currency formatting and casts 'Total' as a decimal to 
      strictly enforce Total > 250.
   5. NON-PUBLIC OFFICIAL FILTER: Strictly filters for 'N' or 'NO'.
   6. TARGET CATEGORY CODES: Hardcoded list of 29 specific category codes.
   7. THE 793 EXCEPTION RULE: If the Category Code is '793', it is IGNORED unless 
      the mapped AU is strictly '101016' (TD Asset Management).
   8. FINAL OUTPUT: Returns the count of matching transactions per AU, anchored 
      to the Master List (defaults to '0' if no matches).
=================================================================================== */

WITH Master_AUs AS (
    -- 1. Grab Master List data: This is our strict anchor for the final output
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

Combined_Coupa_Raw AS (
    -- 2. Union all 7 Coupa files into one master dataset, including the Total column
    SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_jan_feb2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_mar_april2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_may2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_june2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_jul_aug2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_sep_oct2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_nov_dec2024_filtered
),

Coupa_Parsed AS (
    -- 3 & 4. Parse the Account string and clean the Total for mathematical comparison
    SELECT 
        TRY_CAST(TRIM(SPLIT(Account, '-')[0]) AS INT) AS Cleaned_CC,
        TRIM(SPLIT(Account, '-')[2]) AS CatCode,
        TRIM(PublicOfficial) AS PublicOfficial,
        -- RegEx removes $ and commas so it can safely cast to a decimal
        CAST(REGEXP_REPLACE(Total, '[^0-9\\.-]', '') AS DECIMAL(10,2)) AS Numeric_Total
    FROM Combined_Coupa_Raw
    WHERE Account LIKE '%-%-%'
),

CC_Mapping AS (
    -- Standardize the CC Mapping table from our bootstrap
    SELECT DISTINCT 
        TRIM(CAST(AU_ID AS STRING)) AS AU_ID,
        TRY_CAST(TRIM(cost_center_id) AS INT) AS Mapped_CC
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL AND cost_center_id IS NOT NULL
),

Flagged_Transactions AS (
    -- 5, 6, 7. Apply all Business Rules to filter valid transactions
    SELECT 
        c.Cleaned_CC,
        c.CatCode,
        m.AU_ID
    FROM Coupa_Parsed c
    INNER JOIN CC_Mapping m
        ON c.Cleaned_CC = m.Mapped_CC
    WHERE 
        -- RULE: Must NOT be a Public Official
        UPPER(TRIM(c.PublicOfficial)) IN ('N', 'NO')
        
        -- RULE: Threshold must be strictly greater than 250
        AND c.Numeric_Total > 250
        
        -- RULE: Must be one of the target category codes
        AND c.CatCode IN (
            '066', '009', '012', '067', '095', '134', '168', '192', '203', '204', 
            '208', '209', '242', '269', '270', '484', '487', '636', '637', '638', 
            '639', '676', '782', '783', '784', '792', '793', '890', '892'
        )
        
        -- RULE: The 793 Exception Rule (Ignores 793 unless the mapped AU is 101016)
        AND (c.CatCode != '793' OR (c.CatCode = '793' AND m.AU_ID = '101016'))
),

Flagged_AUs AS (
    -- Map flagged transactions to their AU_IDs and COUNT them
    SELECT 
        AU_ID,
        COUNT(*) AS Flagged_Count
    FROM Flagged_Transactions 
    GROUP BY AU_ID
)

-- 8. Final Output: Strictly structured to 6 columns, anchored to the Master list
SELECT 
    a.BusinessID,                          
    a.AU_Name,                             
    a.Business_Segment,
    'EBA04' AS QuestionID,               
    COALESCE(CAST(f.Flagged_Count AS STRING), '0') AS Response,
    a.In_ABAC_AU_List
FROM Master_AUs a
LEFT JOIN Flagged_AUs f
    ON a.BusinessID = f.AU_ID
ORDER BY a.BusinessID;

In [ ]:
%sql
WITH Combined_Coupa_Raw AS (
    SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_jan_feb2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_mar_april2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_may2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_june2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_jul_aug2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_sep_oct2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_nov_dec2024_filtered
),

Coupa_Parsed AS (
    SELECT 
        Account AS Raw_Account,
        TRY_CAST(TRIM(SPLIT(Account, '-')[0]) AS INT) AS Cleaned_CC,
        TRIM(SPLIT(Account, '-')[2]) AS CatCode,
        TRIM(PublicOfficial) AS PublicOfficial,
        Total AS Raw_Total_String,
        CAST(REGEXP_REPLACE(Total, '[^0-9\\.-]', '') AS DECIMAL(10,2)) AS Numeric_Total
    FROM Combined_Coupa_Raw
    WHERE Account LIKE '%-%-%'
),

CC_Mapping AS (
    SELECT DISTINCT
        TRIM(CAST(AU_ID AS STRING)) AS AU_ID,
        TRIM(AU_Name) AS AU_Name,
        TRY_CAST(TRIM(cost_center_id) AS INT) AS Mapped_CC
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL AND cost_center_id IS NOT NULL
),

Mapped_To_AU AS (
    SELECT 
        c.*,
        m.AU_ID,
        m.AU_Name
    FROM Coupa_Parsed c
    LEFT JOIN CC_Mapping m
        ON c.Cleaned_CC = m.Mapped_CC
)

SELECT 
    Raw_Account,
    Cleaned_CC AS Cost_Center_ID,
    CatCode,
    PublicOfficial,
    Raw_Total_String,
    Numeric_Total,
    AU_ID AS Bridged_Master_AU,
    AU_Name AS Bridged_AU_Name,

    -- RULE-BY-RULE CHECKS
    CASE 
        WHEN AU_ID IS NULL THEN 'FAIL: CC failed to bridge to Master AU'
        ELSE 'Pass'
    END AS Bridge_Check,
    CASE 
        WHEN UPPER(TRIM(PublicOfficial)) IN ('N', 'NO') THEN 'Pass'
        ELSE 'FAIL: Not flagged as Non-PO (N/NO)'
    END AS PublicOfficial_Check,
    CASE 
        WHEN Numeric_Total > 250 THEN 'Pass (> 250)'
        ELSE 'FAIL (<= 250)'
    END AS Threshold_Check,
    CASE 
        WHEN CatCode IN ('066', '009', '012', '067', '095', '134', '168', '192', '203', '204', '208', '209', '242', '269', '270', '484', '487', '636', '637', '638', '639', '676', '782', '783', '784', '792', '793', '890', '892') THEN 'Pass'
        ELSE 'FAIL: Invalid Category Code'
    END AS CatCode_Check,
    CASE 
        WHEN CatCode = '793' AND AU_ID != '101016' THEN 'FAIL: Cat 793 strictly locked to 101016'
        ELSE 'Pass'
    END AS Exception_793_Check,

    -- FINAL DECISION (THIS DRIVES EBA04 NUMERIC RESPONSE)
    CASE 
        WHEN AU_ID IS NOT NULL
         AND UPPER(TRIM(PublicOfficial)) IN ('N', 'NO')
         AND Numeric_Total > 250
         AND CatCode IN ('066', '009', '012', '067', '095', '134', '168', '192', '203', '204', '208', '209', '242', '269', '270', '484', '487', '636', '637', '638', '639', '676', '782', '783', '784', '792', '793', '890', '892')
         AND NOT (CatCode = '793' AND AU_ID != '101016')
        THEN 'YES'
        ELSE 'NO'
    END AS Final_Decision,
    CASE 
        WHEN AU_ID IS NOT NULL
         AND UPPER(TRIM(PublicOfficial)) IN ('N', 'NO')
         AND Numeric_Total > 250
         AND CatCode IN ('066', '009', '012', '067', '095', '134', '168', '192', '203', '204', '208', '209', '242', '269', '270', '484', '487', '636', '637', '638', '639', '676', '782', '783', '784', '792', '793', '890', '892')
         AND NOT (CatCode = '793' AND AU_ID != '101016')
        THEN CONCAT(
            'YES because AU ', AU_ID,
            ' is bridged, Non-PO flag=', UPPER(TRIM(PublicOfficial)),
            ', Total=', CAST(Numeric_Total AS STRING),
            ', CatCode=', CatCode,
            CASE WHEN CatCode = '793' THEN ' (valid only for AU 101016)' ELSE '' END
        )
        ELSE NULL
    END AS Why_Yes,
    CASE 
        WHEN AU_ID IS NULL THEN 'NO because CC failed to bridge to Master AU'
        WHEN UPPER(TRIM(PublicOfficial)) NOT IN ('N', 'NO') THEN 'NO because row is not Non-PO (N/NO)'
        WHEN Numeric_Total <= 250 THEN 'NO because Total is <= 250 threshold'
        WHEN CatCode NOT IN ('066', '009', '012', '067', '095', '134', '168', '192', '203', '204', '208', '209', '242', '269', '270', '484', '487', '636', '637', '638', '639', '676', '782', '783', '784', '792', '793', '890', '892') THEN 'NO because Category Code is outside EBA04 list'
        WHEN CatCode = '793' AND AU_ID != '101016' THEN 'NO because CatCode 793 is valid only for AU 101016'
        ELSE NULL
    END AS Why_No

FROM Mapped_To_AU
-- Optional: Uncomment below to only see rows that contribute to EBA04 counted instances
-- WHERE AU_ID IS NOT NULL
--   AND UPPER(TRIM(PublicOfficial)) IN ('N', 'NO')
--   AND Numeric_Total > 250
--   AND CatCode IN ('066', '009', '012', '067', '095', '134', '168', '192', '203', '204', '208', '209', '242', '269', '270', '484', '487', '636', '637', '638', '639', '676', '782', '783', '784', '792', '793', '890', '892')
--   AND NOT (CatCode = '793' AND AU_ID != '101016')
ORDER BY Cost_Center_ID;